In [1]:
import torch
import time
import concurrent.futures
from pathlib import Path
from io import BytesIO

from pdf2image import convert_from_path
import pytesseract
from pypdf import PdfReader, PdfWriter

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# --- NEW LANGGRAPH & AGENT IMPORTS ---
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

In [7]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import torch

# 1. Load the main document (DO NOT load the TOC_Details.pdf into the DB)
loader = PyPDFLoader("/Users/saketh/PennyStockLLM/Penny_stock_PDFS/ssrn_id3089664_code1897729.pdf")
docs = loader.load()

# 2. Define the TOC mapping based on TOC_Details.pdf
def get_chapter_for_page(page_index):
    actual_page = page_index + 1
    
    if actual_page < 4:
        return "Chapter 1: INTRODUCTION"
    elif actual_page < 7:
        return "Chapter 2: LITERATURE REVIEW"
    elif actual_page < 14:
        return "Chapter 3: DATA"
    elif actual_page < 16:
        return "Chapter 4: METHODOLOGY"
    elif actual_page < 25:
        return "Chapter 5: CHARACTERISTICS OF PENNY STOCKS"
    elif actual_page < 27:
        return "Chapter 6: SUMMARY STATISTICS"
    elif actual_page < 30:
        return "Chapter 7: ASSET PRICING REGRESSION RESULTS"
    elif actual_page < 31:
        return "Chapter 8: TAKEAWAYS"
    elif actual_page < 32:
        return "Chapter 9: SCOPE FOR FURTHER RESEARCH"
    elif actual_page < 34:
        return "REFERENCES"
    else:
        return "APPENDIX"

# 3. Inject the metadata into each page BEFORE chunking
for doc in docs:
    # PyPDFLoader automatically extracts the page number into metadata['page']
    page_index = doc.metadata.get("page", 0) 
    # Create a new metadata key called 'chapter'
    doc.metadata["chapter"] = get_chapter_for_page(page_index)

# 4. Split the documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400,
    add_start_index=True,
)

# The splitter automatically copies the "chapter" metadata to every small chunk it creates
chunks = text_splitter.split_documents(docs)
print(f"Generated {len(chunks)} chunks with chapter metadata.")


# ==========================================
# NEW EMBEDDING AND DATABASE STORAGE CODE
# ==========================================

# 5. Initialize GPU-accelerated embeddings
print("Initializing GPU-accelerated embeddings...")
device = "mps" if torch.backends.mps.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)

# 6. Store in ChromaDB
DB_DIR = "./penny_stock_vector_db"
COLLECTION_NAME = "ssrn_id3089664" # Unique drawer inside the cabinet

print(f"Storing into collection: '{COLLECTION_NAME}'...")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR,
    collection_name=COLLECTION_NAME,
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"Success! Embedded {len(chunks)} chunks into {DB_DIR} under the collection '{COLLECTION_NAME}'.")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 94 0 (offset 0)
Ignoring wrong pointing object 141 0 (offset 0)
Ignoring wrong pointing object 143 0 (offset 0)
Ignoring wrong pointing object 146 0 (offset 0)
Ignoring wrong pointing object 148 0 (offset 0)
Ignoring wrong pointing object 150 0 (offset 0)
Ignoring wrong pointing object 152 0 (offset 0)
Ignoring wrong pointing object 157 0 (offset 0)
Ignoring wrong pointing object 159 0 (offset 0)
Ignoring wrong pointing object 161 0 (offset 0)
Ignoring wrong pointing object 163 0 (offset 0)
Ignoring wrong pointing object 168 0 (offset 0)
Igno

Generated 89 chunks with chapter metadata.
Initializing GPU-accelerated embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Storing into collection: 'ssrn_id3089664'...
Success! Embedded 89 chunks into ./penny_stock_vector_db under the collection 'ssrn_id3089664'.


In [8]:
import chromadb
print(chromadb.PersistentClient(path="./penny_stock_vector_db").list_collections())

[Collection(name=langchain_large), Collection(name=ssrn_id3089664), Collection(name=langchain)]


In [5]:
import time
import concurrent.futures
from pathlib import Path
from io import BytesIO

from pdf2image import convert_from_path
import pytesseract
from pypdf import PdfReader, PdfWriter

# --- 1. Modified Worker Function ---
def process_single_page(image):
    """
    Returns RAW BYTES instead of a complex PDF object.
    Raw bytes can be safely passed between CPU cores without crashing.
    """
    return pytesseract.image_to_pdf_or_hocr(image, extension="pdf")

# --- 2. Main Execution ---
if __name__ == '__main__':
    input_pdf = Path("/Users/saketh/PennyStockLLM/Penny_stock.pdf")
    output_pdf = Path("Penny_stock_ocr.pdf")
    writer = PdfWriter()

    start_all = time.perf_counter()
    print("Extracting images from PDF (this might take a moment)...")
    
    # Extract images using threads
    images = convert_from_path(str(input_pdf), dpi=300, thread_count=12)
    print(f"Extracted {len(images)} images.")

    print(f"Starting Multi-core OCR processing on {len(images)} pages...")
    
    # Use threads because PIL image objects cannot be reliably pickled across processes
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        ocr_byte_results = list(executor.map(process_single_page, images))

    print("Stitching PDF back together in the main process...")
    # Now we parse those bytes into PDF pages safely in the main thread
    for idx, pdf_bytes in enumerate(ocr_byte_results, start=1):
        reader = PdfReader(BytesIO(pdf_bytes))
        writer.add_page(reader.pages[0])
        
        # Print a progress update every 10 pages
        if idx % 10 == 0 or idx == len(ocr_byte_results):
            print(f"Stitched {idx}/{len(ocr_byte_results)} pages")

    print("Saving file to disk...")
    with output_pdf.open("wb") as f:
        writer.write(f)

    print("Saved searchable PDF:", output_pdf)

    total_elapsed = time.perf_counter() - start_all
    print(f"Total time: {total_elapsed:.2f} seconds")

Extracting images from PDF (this might take a moment)...
Extracted 205 images.
Starting Multi-core OCR processing on 205 pages...
Stitching PDF back together in the main process...
Stitched 10/205 pages
Stitched 20/205 pages
Stitched 30/205 pages
Stitched 40/205 pages
Stitched 50/205 pages
Stitched 60/205 pages
Stitched 70/205 pages
Stitched 80/205 pages
Stitched 90/205 pages
Stitched 100/205 pages
Stitched 110/205 pages
Stitched 120/205 pages
Stitched 130/205 pages
Stitched 140/205 pages
Stitched 150/205 pages
Stitched 160/205 pages
Stitched 170/205 pages
Stitched 180/205 pages
Stitched 190/205 pages
Stitched 200/205 pages
Stitched 205/205 pages
Saving file to disk...
Saved searchable PDF: Penny_stock_ocr.pdf
Total time: 65.59 seconds


In [6]:
#Embeddings process

PDF_PATH = "Penny_stock_ocr.pdf"
DB_DIR = "./penny_stock_vector_db"

print("Loading PDF...")
loader = PyPDFLoader(PDF_PATH)
# Use fast mode by avoiding unnecessary metadata parsing
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 400,
    add_start_index = True,
)

chunks = text_splitter.split_documents(docs)
print(f"Generated {len(chunks)} semantic chunks.")

# 3. GPU-ACCELERATED EMBEDDINGS
print("Initializing GPU-accelerated embeddings...")

device = "mps" if torch.backends.mps.is_available() else "cpu"

model_kwargs = {'device': device}
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5",
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs
)

# 4. BATCHED VECTOR STORAGE
print("Storing in ChromaDB (using GPU)...")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR,
    collection_name="langchain_large",
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"Optimization Complete! Vector DB saved at: {DB_DIR}")



Loading PDF...
Generated 206 semantic chunks.
Initializing GPU-accelerated embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Storing in ChromaDB (using GPU)...
Optimization Complete! Vector DB saved at: ./penny_stock_vector_db


In [ ]:
# 1. ESTABLISH CONNECTION TO EMBEDDINGS & DATABASE
print("Connecting to local Vector DB...")
device = "mps" if torch.backends.mps.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5",
    model_kwargs={'device':device},
    encode_kwargs={'normalize_embeddings': True}
)

vector_db = Chroma(
    persist_directory="./penny_stock_vector_db",
    embedding_function=embeddings,
    collection_name="langchain_large"
)

# 2. SETUP RETRIEVER
retriever = vector_db.as_retriever(search_type="mmr", search_kwargs={"k": 8, "fetch_k": 30})

# 3. DEFINE THE AGENT'S TOOLS
@tool
def penny_stock_researcher(query: str) -> str:
    """Searches the local vector database for SEC filings, news, and 
    historical data related to penny stocks. Use this for fundamental analysis."""
    print(f"\n[Agent Action] -> Querying Vector DB for: {query}")
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

@tool
def real_time_price_checker(ticker: str) -> str:
    """Fetches the current market price for a given stock ticker symbol."""
    print(f"\n[Agent Action] -> Checking real-time price for: {ticker}")
    # Dummy data for now - we will connect an API like yfinance here later!
    return f"The current price of {ticker} is $0.45 (Up 5% today)."

tools = [penny_stock_researcher, real_time_price_checker]

# 4. INITIALIZE THE LLM AS A SUPERVISOR
print("Waking up Qwen 3.5 (35B) as an Agent...")
# We use ChatOllama and bind the tools so the LLM knows what its "hands" can do
llm = ChatOllama(model="qwen3.5:35b", temperature=0).bind_tools(tools)

# 5. DEFINE THE STATE AND GRAPH WORKFLOW
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def call_model(state: AgentState):
    """Invokes the LLM to decide the next action or synthesize the final answer."""
    response = llm.invoke(state['messages'])
    return {"messages": [response]}

def should_continue(state: AgentState):
    """Router logic: checks if the LLM requested a tool."""
    last_message = state['messages'][-1]
    if not last_message.tool_calls:
        return "end"
    return "continue"

print("Compiling LangGraph Workflow...")
workflow = StateGraph(AgentState)

# Add the nodes (the workers)
workflow.add_node("agent", call_model)
workflow.add_node("action", ToolNode(tools))

# Set the starting point
workflow.set_entry_point("agent")

# Define the conditional routing logic
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "action",
        "end": END
    }
)

# After an action (tool) is completed, ALWAYS loop back to the agent for synthesis
workflow.add_edge("action", "agent")

# Compile the final application
app = workflow.compile()

# 6. EXECUTE THE MULTI-AGENT PIPELINE
print("\n--- INITIATING AI AGENT ---")
query = "I have Rs 100. Based on the documents, is buying a penny stock like Ticker XYZ a safe investment compared to large-cap stocks? Also, what is the current price of XYZ?"

inputs = {"messages": [HumanMessage(content=query)]}

# Stream the agent's thought process
for event in app.stream(inputs, stream_mode="values"):
    message = event["messages"][-1]
    if message.content:
        print(f"\n[Agent Output]:\n{message.content}")

Connecting to local Vector DB...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Waking up Qwen 3.5 (35B) as an Agent...
Compiling LangGraph Workflow...

--- INITIATING AI AGENT ---

[Agent Output]:
I have Rs 100. Based on the documents, is buying a penny stock like Ticker XYZ a safe investment compared to large-cap stocks? Also, what is the current price of XYZ?

[Agent Action] -> Querying Vector DB for: penny stock safety comparison large-cap stocks investment risk
[Agent Action] -> Checking real-time price for: XYZ


[Agent Output]:
The current price of XYZ is $0.45 (Up 5% today).

[Agent Output]:
Based on the documents provided and the current market data:

**Is buying a penny stock like XYZ a safe investment compared to large-cap stocks?**

**No, it is generally not considered a safe investment.** The documents highlight several key risks associated with penny stocks (often categorized as micro-cap or small-cap) compared to large-cap stocks:

*   **High Risk of Wealth Destruction:** The text explicitly warns that penny stocks can be "wealth destroyers." It cit